# CBORチュートリアル
## Encoder / Decoder を一から組み立てる

このノートブックでは、CBOR（Concise Binary Object Representation）の仕様を読み解きながら、
**cbor2** ライブラリのアーキテクチャに沿って Encoder / Decoder を段階的に実装します。

---

### CBORとは

CBORは [RFC 8949](https://www.rfc-editor.org/rfc/rfc8949) で定義されたバイナリのエンコーディングフォーマットです。
JSONに似た構造を持ちつつ、**コンパクト・高速・スキーマ不要**という特徴があります。

| 形式 | 例 (`{"a":1}`) | サイズ |
|------|-----------------|--------|
| JSON | `{"a":1}` (文字列) | 7 bytes |
| CBOR | バイナリ | 4 bytes |

### CBORデータの構造：すべては「先頭バイト（initial byte）」から始まる

```
  bits: 7 6 5 | 4 3 2 1 0
  major type  | additional info
```

| Major Type | 値 | 意味 |
|------------|-----|------|
| 0 | 0b000 | 符号なし整数 |
| 1 | 0b001 | 負の整数 |
| 2 | 0b010 | バイト列 |
| 3 | 0b011 | テキスト文字列 |
| 4 | 0b100 | 配列 |
| 5 | 0b101 | マップ |
| 6 | 0b110 | セマンティックタグ |
| 7 | 0b111 | 浮動小数点・特殊値 |

---
## Step 1

In [2]:
# CBORTag
# CBORの「セマンティックタグ」を表すクラス。
# タグ番号（整数）と値のペアで、その値の「意味」を付与します。
# 例: tag=0 → datetime文字列, tag=1 → epoch timestamp

from functools import total_ordering

@total_ordering
class CBORTag:
    """CBORセマンティックタグ (tag番号 + 値)"""
    __slots__ = ('tag', 'value')

    def __init__(self, tag: int, value):
        if not isinstance(tag, int) or not (0 <= tag < 2**64):
            raise TypeError("tag は 0 以上 2**64 未満の整数である必要があります")
        self.tag = tag
        self.value = value

    def __eq__(self, other):
        if isinstance(other, CBORTag):
            return (self.tag, self.value) == (other.tag, other.value)
        return NotImplemented

    def __le__(self, other):
        if isinstance(other, CBORTag):
            return (self.tag, self.value) <= (other.tag, other.value)
        return NotImplemented

    def __repr__(self):
        return f'CBORTag({self.tag}, {self.value!r})'

    def __hash__(self):
        return hash((self.tag, self.value))


# 動作確認
t = CBORTag(1, 1700000000.0)   # epoch timestamp タグ
print(t)
print(f"tag={t.tag}, value={t.value}")

CBORTag(1, 1700000000.0)
tag=1, value=1700000000.0


In [4]:
# CBORSimpleValue
# CBOR Major Type 7 の「シンプル値」。
# True/False/None/undefined 以外の特殊値を表します。

from collections import namedtuple

class CBORSimpleValue(namedtuple('CBORSimpleValue', ['value'])):
    """CBORシンプル値 (0-255, ただし 24-31 は予約済み)"""
    __slots__ = ()

    def __new__(cls, value: int):
        if value < 0 or value > 255 or (23 < value < 32):
            raise TypeError("値の範囲: 0..23, 32..255")
        return super().__new__(cls, value)

    def __eq__(self, other):
        if isinstance(other, int):
            return self.value == other
        if isinstance(other, CBORSimpleValue):
            return self.value == other.value
        return NotImplemented

    def __repr__(self):
        return f'CBORSimpleValue({self.value})'


# undefined / break_marker
# CBORには「Undefined (0xF7)」と不定長エンコーディングの終端「break (0xFF)」があります。

class _Undefined:
    """CBORの undefined 値 (シングルトン)"""
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    def __repr__(self): return 'undefined'
    def __bool__(self): return False

class _BreakMarker:
    """不定長コンテナの終端マーカー (シングルトン)"""
    _instance = None
    def __new__(cls):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
        return cls._instance
    def __repr__(self): return 'break_marker'

undefined = _Undefined()
break_marker = _BreakMarker()

print(f"undefined: {undefined!r}, bool={bool(undefined)}")
print(f"CBORSimpleValue(16): {CBORSimpleValue(16)}")

undefined: undefined, bool=False
CBORSimpleValue(16): CBORSimpleValue(16)


In [17]:
# ── Chapter 1-D: FrozenDict ───────────────────────────────────────────────────
# ハッシュ可能な不変マッピング型。
# decodeのimutableモードでdictのかわりに使われます。

from collections.abc import Mapping, Iterable, Iterator

class FrozenDict(Mapping):
    """ハッシュ可能なイミュータブルな辞書"""

    def __init__(self, *args):
        self._d = dict(*args)
        self._hash = None

    def __iter__(self) -> Iterator: return iter(self._d)
    def __len__(self) -> int: return len(self._d)
    def __getitem__(self, key): return self._d[key]
    def __repr__(self): return f'FrozenDict({self._d})'

    def __hash__(self) -> int:
        if self._hash is None:
            self._hash = hash((frozenset(self), frozenset(self.values())))
        return self._hash


fd = FrozenDict({'x': 1, 'y': 2})
print(fd)
print(f"ハッシュ可能: {hash(fd)}")
print(f"辞書として使用: {fd['x']}")

FrozenDict({'x': 1, 'y': 2})
ハッシュ可能: 4625798956757667997
辞書として使用: 1


---
## Step 2: Encoder の心臓部 — `encode_length`

CBORのすべてのデータは「initial byte + (追加バイト) + (ペイロード)」で構成されます。

`encode_length` はその initial byte と追加バイトを生成する最重要メソッドです。

```
値の大きさ          | エンコード形式
-------------------+--------------------------------
0 ～ 23           | 1バイト (major << 5 | value)
24 ～ 255         | 2バイト (major << 5 | 24, value)
256 ～ 65535      | 3バイト (major << 5 | 25, uint16)
65536 ～ 2^32-1   | 5バイト (major << 5 | 26, uint32)
2^32 ～ 2^64-1    | 9バイト (major << 5 | 27, uint64)
不定長            | 1バイト (major << 5 | 31)
```

In [18]:
import struct
from io import BytesIO

def encode_length(fp, major_tag: int, length) -> None:
    """major_tag（0-7）と長さ/値をCBOR形式でfpに書き込む"""
    major_tag <<= 5   # 上位3ビットにシフト

    if length is None:           # 不定長
        fp.write(struct.pack('>B', major_tag | 31))
    elif length < 24:            # 追加バイトなし
        fp.write(struct.pack('>B', major_tag | length))
    elif length < 256:           # 1バイト追加
        fp.write(struct.pack('>BB', major_tag | 24, length))
    elif length < 65536:         # 2バイト追加
        fp.write(struct.pack('>BH', major_tag | 25, length))
    elif length < 4294967296:    # 4バイト追加
        fp.write(struct.pack('>BL', major_tag | 26, length))
    else:                        # 8バイト追加
        fp.write(struct.pack('>BQ', major_tag | 27, length))


# ── 可視化ヘルパー ────────────────────────────────────────────────────────────
def show_bytes(data: bytes, label: str = ""):
    hex_str = ' '.join(f'{b:02x}' for b in data)
    bin_str = ' '.join(f'{b:08b}' for b in data)
    print(f"{label}")
    print(f"  hex: [{hex_str}]")
    print(f"  bin: [{bin_str}]")
    print(f"  len: {len(data)} bytes")


# テスト: 整数 0 (major=0, value=0)
buf = BytesIO()
encode_length(buf, 0, 0)
show_bytes(buf.getvalue(), "整数 0")

# テスト: 整数 23 (major=0, value=23) — ギリギリ1バイト
buf = BytesIO()
encode_length(buf, 0, 23)
show_bytes(buf.getvalue(), "整数 23 (1バイト境界)")

# テスト: 整数 24 (major=0) — 2バイトが必要になる境界
buf = BytesIO()
encode_length(buf, 0, 24)
show_bytes(buf.getvalue(), "整数 24 (2バイト境界)")

整数 0
  hex: [00]
  bin: [00000000]
  len: 1 bytes
整数 23 (1バイト境界)
  hex: [17]
  bin: [00010111]
  len: 1 bytes
整数 24 (2バイト境界)
  hex: [18 18]
  bin: [00011000 00011000]
  len: 2 bytes


---
## Step 3: 各型のエンコーダを作る

In [19]:
# 整数エンコーダ
# 正の整数 → major type 0
# 負の整数 → major type 1 (エンコード値は -(n+1))
# 2**64以上の巨大整数 → セマンティックタグ 2/3 でラップ

def encode_int(fp, value: int) -> None:
    """整数をCBORエンコード"""
    if value >= 18446744073709551616 or value < -18446744073709551616:
        # 巨大整数: バイト列としてセマンティックタグで包む
        if value >= 0:
            major_type = 0x02  # tag 2: 正の巨大整数
        else:
            major_type = 0x03  # tag 3: 負の巨大整数
            value = -value - 1
        payload = value.to_bytes((value.bit_length() + 7) // 8, 'big')
        # セマンティックタグとして書き込む（後でencode_semanticとして統合）
        encode_length(fp, 6, major_type)
        encode_length(fp, 2, len(payload))
        fp.write(payload)
    elif value >= 0:
        encode_length(fp, 0, value)
    else:
        encode_length(fp, 1, -(value + 1))


for v in [0, 23, 24, 255, 1000, -1, -100, 2**64]:
    buf = BytesIO()
    encode_int(buf, v)
    show_bytes(buf.getvalue(), f"encode_int({v})")

encode_int(0)
  hex: [00]
  bin: [00000000]
  len: 1 bytes
encode_int(23)
  hex: [17]
  bin: [00010111]
  len: 1 bytes
encode_int(24)
  hex: [18 18]
  bin: [00011000 00011000]
  len: 2 bytes
encode_int(255)
  hex: [18 ff]
  bin: [00011000 11111111]
  len: 2 bytes
encode_int(1000)
  hex: [19 03 e8]
  bin: [00011001 00000011 11101000]
  len: 3 bytes
encode_int(-1)
  hex: [20]
  bin: [00100000]
  len: 1 bytes
encode_int(-100)
  hex: [38 63]
  bin: [00111000 01100011]
  len: 2 bytes
encode_int(18446744073709551616)
  hex: [c2 49 01 00 00 00 00 00 00 00 00]
  bin: [11000010 01001001 00000001 00000000 00000000 00000000 00000000 00000000 00000000 00000000 00000000]
  len: 11 bytes


In [20]:
# 文字列・バイト列エンコーダ
# bytes  → major type 2 (バイト列)
# str    → major type 3 (UTF-8テキスト)

def encode_bytestring(fp, value: bytes) -> None:
    """バイト列をCBORエンコード (major type 2)"""
    encode_length(fp, 2, len(value))
    fp.write(value)

def encode_string(fp, value: str) -> None:
    """テキスト文字列をCBORエンコード (major type 3)"""
    encoded = value.encode('utf-8')
    encode_length(fp, 3, len(encoded))
    fp.write(encoded)


buf = BytesIO()
encode_string(buf, "hello")
show_bytes(buf.getvalue(), 'encode_string("hello")')

buf = BytesIO()
encode_string(buf, "こんにちは")
show_bytes(buf.getvalue(), 'encode_string("こんにちは")')

buf = BytesIO()
encode_bytestring(buf, b"\x01\x02\x03")
show_bytes(buf.getvalue(), 'encode_bytestring(b"\\x01\\x02\\x03")')

encode_string("hello")
  hex: [65 68 65 6c 6c 6f]
  bin: [01100101 01101000 01100101 01101100 01101100 01101111]
  len: 6 bytes
encode_string("こんにちは")
  hex: [6f e3 81 93 e3 82 93 e3 81 ab e3 81 a1 e3 81 af]
  bin: [01101111 11100011 10000001 10010011 11100011 10000010 10010011 11100011 10000001 10101011 11100011 10000001 10100001 11100011 10000001 10101111]
  len: 16 bytes
encode_bytestring(b"\x01\x02\x03")
  hex: [43 01 02 03]
  bin: [01000011 00000001 00000010 00000011]
  len: 4 bytes


In [21]:
# 特殊値エンコーダ (major type 7)
# True=0xF5, False=0xF4, None=0xF6, undefined=0xF7
# float16/32/64 も major type 7

import math

def encode_boolean(fp, value: bool) -> None:
    fp.write(b'\xf5' if value else b'\xf4')

def encode_none(fp, _) -> None:
    fp.write(b'\xf6')

def encode_undefined(fp, _) -> None:
    fp.write(b'\xf7')

def encode_float(fp, value: float) -> None:
    """float64 (double) としてエンコード"""
    if math.isnan(value):
        fp.write(b'\xf9\x7e\x00')   # float16 NaN
    elif math.isinf(value):
        fp.write(b'\xf9\x7c\x00' if value > 0 else b'\xf9\xfc\x00')
    else:
        fp.write(struct.pack('>Bd', 0xFB, value))


for label, fn, val in [
    ('True', encode_boolean, True),
    ('False', encode_boolean, False),
    ('None', encode_none, None),
    ('undefined', encode_undefined, undefined),
    ('3.14', encode_float, 3.14),
    ('inf', encode_float, float('inf')),
    ('nan', encode_float, float('nan')),
]:
    buf = BytesIO()
    fn(buf, val)
    show_bytes(buf.getvalue(), label)

True
  hex: [f5]
  bin: [11110101]
  len: 1 bytes
False
  hex: [f4]
  bin: [11110100]
  len: 1 bytes
None
  hex: [f6]
  bin: [11110110]
  len: 1 bytes
undefined
  hex: [f7]
  bin: [11110111]
  len: 1 bytes
3.14
  hex: [fb 40 09 1e b8 51 eb 85 1f]
  bin: [11111011 01000000 00001001 00011110 10111000 01010001 11101011 10000101 00011111]
  len: 9 bytes
inf
  hex: [f9 7c 00]
  bin: [11111001 01111100 00000000]
  len: 3 bytes
nan
  hex: [f9 7e 00]
  bin: [11111001 01111110 00000000]
  len: 3 bytes


---
## Step 4: CBOREncoder クラスに組み上げる

ここまでの部品をクラスにまとめ、**型ディスパッチ**（型→エンコーダ関数のマッピング）を実装します。

In [34]:
from collections import OrderedDict, defaultdict
from collections.abc import Mapping, Sequence
import struct, math
from io import BytesIO
from typing import Any, Callable, Optional


class MyCBOREncoder:
    def __init__(self, fp, default: Optional[Callable] = None):
        self._fp = fp
        self._default = default
        # 型 → エンコーダ関数のマッピング
        self._encoders: dict = {
            bool:         self._encode_boolean,   # bool は int のサブクラスなので先に登録
            int:          self._encode_int,
            float:        self._encode_float,
            bytes:        self._encode_bytestring,
            bytearray:    self._encode_bytearray,
            str:          self._encode_string,
            list:         self._encode_array,
            tuple:        self._encode_array,
            dict:         self._encode_map,
            defaultdict:  self._encode_map,
            OrderedDict:  self._encode_map,
            FrozenDict:   self._encode_map,
            type(None):   self._encode_none,
            type(undefined): self._encode_undefined,
            CBORTag:      self._encode_semantic,
            CBORSimpleValue: self._encode_simple_value,
            set:          self._encode_set,
            frozenset:    self._encode_set,
        }

    # ── 内部ヘルパー ─────────────────────────────────────────────────────────

    def _write(self, data: bytes):
        self._fp.write(data)

    def _encode_length(self, major_tag: int, length) -> None:
        major_tag <<= 5
        if length is None:
            self._write(struct.pack('>B', major_tag | 31))
        elif length < 24:
            self._write(struct.pack('>B', major_tag | length))
        elif length < 256:
            self._write(struct.pack('>BB', major_tag | 24, length))
        elif length < 65536:
            self._write(struct.pack('>BH', major_tag | 25, length))
        elif length < 4294967296:
            self._write(struct.pack('>BL', major_tag | 26, length))
        else:
            self._write(struct.pack('>BQ', major_tag | 27, length))

    # ── 型別エンコーダ ────────────────────────────────────────────────────────

    def _encode_int(self, value: int) -> None:
        if value >= 18446744073709551616 or value < -18446744073709551616:
            if value >= 0:
                mt = 0x02
            else:
                mt = 0x03
                value = -value - 1
            payload = value.to_bytes((value.bit_length() + 7) // 8, 'big')
            self._encode_semantic(CBORTag(mt, payload))
        elif value >= 0:
            self._encode_length(0, value)
        else:
            self._encode_length(1, -(value + 1))

    def _encode_boolean(self, value: bool) -> None:
        self._write(b'\xf5' if value else b'\xf4')

    def _encode_none(self, value) -> None:
        self._write(b'\xf6')

    def _encode_undefined(self, value) -> None:
        self._write(b'\xf7')

    def _encode_float(self, value: float) -> None:
        if math.isnan(value):
            self._write(b'\xf9\x7e\x00')
        elif math.isinf(value):
            self._write(b'\xf9\x7c\x00' if value > 0 else b'\xf9\xfc\x00')
        else:
            self._write(struct.pack('>Bd', 0xFB, value))

    def _encode_bytestring(self, value: bytes) -> None:
        self._encode_length(2, len(value))
        self._write(value)

    def _encode_bytearray(self, value: bytearray) -> None:
        self._encode_bytestring(bytes(value))

    def _encode_string(self, value: str) -> None:
        encoded = value.encode('utf-8')
        self._encode_length(3, len(encoded))
        self._write(encoded)

    def _encode_array(self, value) -> None:
        self._encode_length(4, len(value))
        for item in value:
            self._encode_value(item)

    def _encode_map(self, value: Mapping) -> None:
        self._encode_length(5, len(value))
        for k, v in value.items():
            self._encode_value(k)
            self._encode_value(v)

    def _encode_semantic(self, value: CBORTag) -> None:
        """セマンティックタグ (major type 6) をエンコード"""
        self._encode_length(6, value.tag)
        self._encode_value(value.value)

    def _encode_simple_value(self, value: CBORSimpleValue) -> None:
        if value.value < 24:
            self._write(struct.pack('>B', 0xE0 | value.value))
        else:
            self._write(struct.pack('>BB', 0xF8, value.value))

    def _encode_set(self, value) -> None:
        """set/frozenset はセマンティックタグ258でラップ"""
        self._encode_semantic(CBORTag(258, tuple(value)))

    # ── メインエントリポイント ────────────────────────────────────────────────

    def _encode_value(self, obj: Any) -> None:
        """型ディスパッチして適切なエンコーダを呼ぶ"""
        obj_type = obj.__class__
        encoder = self._encoders.get(obj_type)

        if encoder is None:
            # サブクラス対応: issubclassで探す
            for t, enc in self._encoders.items():
                if isinstance(obj, t):
                    self._encoders[obj_type] = enc  # キャッシュ
                    encoder = enc
                    break

        if encoder is None:
            if self._default:
                self._default(self, obj)
                return
            raise TypeError(f"エンコード不可能な型: {obj_type.__name__}")

        encoder(obj)

    def encode(self, obj: Any) -> None:
        """オブジェクトをCBORエンコードしてfpに書き込む"""
        self._encode_value(obj)


def my_dumps(obj: Any, **kwargs) -> bytes:
    """オブジェクトをCBORバイト列に変換"""
    buf = BytesIO()
    MyCBOREncoder(buf, **kwargs).encode(obj)
    return buf.getvalue()


# ── 動作確認 ──────────────────────────────────────────────────────────────────
test_cases = [
    0, 23, 24, 255, 1000, -1, -100,
    True, False, None, undefined,
    3.14, float('inf'), float('nan'),
    "hello", "こんにちは",
    b"\x01\x02",
    [1, 2, 3],
    {"key": "value", "num": 42},
    CBORTag(1, 1700000000),
    {1, 2, 3},
]

for obj in test_cases:
    encoded = my_dumps(obj)
    print(f"{repr(obj):<30} → {encoded.hex():<40} ({len(encoded)} bytes)")

0                              → 00                                       (1 bytes)
23                             → 17                                       (1 bytes)
24                             → 1818                                     (2 bytes)
255                            → 18ff                                     (2 bytes)
1000                           → 1903e8                                   (3 bytes)
-1                             → 20                                       (1 bytes)
-100                           → 3863                                     (2 bytes)
True                           → f5                                       (1 bytes)
False                          → f4                                       (1 bytes)
None                           → f6                                       (1 bytes)
undefined                      → f7                                       (1 bytes)
3.14                           → fb40091eb851eb851f                       (9

---
## Step 5: Decoder の心臓部 — initial byte の解析

デコーダは逆方向の処理です。バイト列を読んで、
1. initial byte から major type と additional info を取り出す
2. additional info に応じて長さ/値を読む
3. ペイロードを読んでPythonオブジェクトを構築する

In [23]:
# _decode_length
# Encoderのencode_lengthの逆操作。
# subtypeはinitial byteの下位5ビット。

import struct
from io import BytesIO

def _read(fp, amount: int) -> bytes:
    data = fp.read(amount)
    if len(data) < amount:
        raise EOFError(f"ストリーム終端 (期待: {amount}バイト, 実際: {len(data)}バイト)")
    return data

def _decode_length(fp, subtype: int, allow_indefinite: bool = False):
    """additional info (subtype) から長さ/値を読む"""
    if subtype < 24:
        return subtype                                          # 値が即値
    elif subtype == 24:
        return _read(fp, 1)[0]                                  # 1バイト追加
    elif subtype == 25:
        return struct.unpack('>H', _read(fp, 2))[0]            # 2バイト追加
    elif subtype == 26:
        return struct.unpack('>L', _read(fp, 4))[0]            # 4バイト追加
    elif subtype == 27:
        return struct.unpack('>Q', _read(fp, 8))[0]            # 8バイト追加
    elif subtype == 31 and allow_indefinite:
        return None                                             # 不定長
    else:
        raise ValueError(f"不明なsubtype: 0x{subtype:x}")


# CBORバイト列を手動で解析する例
# 0x18 0x64 → major=0(uint), subtype=24(1バイト追加) → 0x64=100
buf = BytesIO(bytes([0x18, 0x64]))
initial = buf.read(1)[0]
major = initial >> 5
sub   = initial & 0x1F
value = _decode_length(buf, sub)
print(f"initial=0x{initial:02x}, major={major}, subtype={sub}, value={value}")

initial=0x18, major=0, subtype=24, value=100


---
## Step 6: MyCBORDecoder クラスに組み上げる

In [33]:
import re, struct
from datetime import datetime, timezone, timedelta
from io import BytesIO

timestamp_re = re.compile(
    r'^(\d{4})-(\d\d)-(\d\d)T(\d\d):(\d\d):(\d\d)'
    r'(?:\.(\d{1,6})\d*)?(?:Z|([+-])(\d\d):(\d\d))$'
)


class MyCBORDecoder:
    def __init__(self, fp, tag_hook=None, object_hook=None):
        self._fp = fp
        self._tag_hook = tag_hook
        self._object_hook = object_hook
        self._immutable = False

        # major type → デコーダメソッドのマッピング
        self._major_decoders = {
            0: self._decode_uint,
            1: self._decode_negint,
            2: self._decode_bytestring,
            3: self._decode_string,
            4: self._decode_array,
            5: self._decode_map,
            6: self._decode_semantic,
            7: self._decode_special,
        }

        # セマンティックタグ番号 → デコーダメソッドのマッピング
        self._semantic_decoders = {
            0:   self._decode_datetime_string,
            1:   self._decode_epoch_datetime,
            2:   self._decode_positive_bignum,
            3:   self._decode_negative_bignum,
            258: self._decode_set,
        }

        # special subtype → デコーダ
        self._special_decoders = {
            20: lambda: False,
            21: lambda: True,
            22: lambda: None,
            23: lambda: undefined,
            25: self._decode_float16,
            26: self._decode_float32,
            27: self._decode_float64,
            31: lambda: break_marker,
        }

    # ── 内部ヘルパー ─────────────────────────────────────────────────────────

    def _read(self, amount: int) -> bytes:
        data = self._fp.read(amount)
        if len(data) < amount:
            raise EOFError(f"ストリーム終端 (期待: {amount}バイト)")
        return data

    def _decode_length(self, subtype: int, allow_indefinite=False):
        if subtype < 24:   return subtype
        elif subtype == 24: return self._read(1)[0]
        elif subtype == 25: return struct.unpack('>H', self._read(2))[0]
        elif subtype == 26: return struct.unpack('>L', self._read(4))[0]
        elif subtype == 27: return struct.unpack('>Q', self._read(8))[0]
        elif subtype == 31 and allow_indefinite: return None
        else: raise ValueError(f"不明なsubtype: 0x{subtype:x}")

    # ── major type デコーダ ───────────────────────────────────────────────────

    def _decode_uint(self, subtype: int) -> int:
        return self._decode_length(subtype)

    def _decode_negint(self, subtype: int) -> int:
        return -self._decode_length(subtype) - 1

    def _decode_bytestring(self, subtype: int) -> bytes:
        length = self._decode_length(subtype, allow_indefinite=True)
        if length is None:
            chunks = []
            while True:
                ib = self._read(1)[0]
                if ib == 0xFF: break
                assert ib >> 5 == 2
                n = self._decode_length(ib & 0x1F)
                chunks.append(self._read(n))
            return b''.join(chunks)
        return self._read(length)

    def _decode_string(self, subtype: int) -> str:
        length = self._decode_length(subtype, allow_indefinite=True)
        if length is None:
            chunks = []
            while True:
                ib = self._read(1)[0]
                if ib == 0xFF: break
                assert ib >> 5 == 3
                n = self._decode_length(ib & 0x1F)
                chunks.append(self._read(n).decode('utf-8'))
            return ''.join(chunks)
        return self._read(length).decode('utf-8')

    def _decode_array(self, subtype: int):
        length = self._decode_length(subtype, allow_indefinite=True)
        items = []
        if length is None:
            while True:
                val = self._decode()
                if val is break_marker: break
                items.append(val)
        else:
            for _ in range(length):
                items.append(self._decode())
        return tuple(items) if self._immutable else items

    def _decode_map(self, subtype: int):
        length = self._decode_length(subtype, allow_indefinite=True)
        d = {}
        if length is None:
            while True:
                old_immutable, self._immutable = self._immutable, True
                key = self._decode()
                self._immutable = old_immutable
                if key is break_marker: break
                d[key] = self._decode()
        else:
            for _ in range(length):
                old_immutable, self._immutable = self._immutable, True
                key = self._decode()
                self._immutable = old_immutable
                d[key] = self._decode()

        if self._object_hook:
            return self._object_hook(self, d)
        if self._immutable:
            return FrozenDict(d)
        return d

    def _decode_semantic(self, subtype: int):
        tagnum = self._decode_length(subtype)
        if dec := self._semantic_decoders.get(tagnum):
            return dec()
        # 未知のタグ
        tag = CBORTag(tagnum, self._decode())
        if self._tag_hook:
            return self._tag_hook(self, tag)
        return tag

    def _decode_special(self, subtype: int):
        if subtype < 20:
            return CBORSimpleValue(subtype)
        if dec := self._special_decoders.get(subtype):
            return dec()
        raise ValueError(f"未サポートのspecialサブタイプ: 0x{subtype:x}")

    # ── セマンティックタグ デコーダ ───────────────────────────────────────────

    def _decode_datetime_string(self):
        value = self._decode()
        m = timestamp_re.match(value)
        if not m:
            raise ValueError(f"不正なdatetimeフォーマット: {value!r}")
        year, month, day, hour, minute, second, frac, sign, oh, om = m.groups()
        us = int(f"{frac:<06}") if frac else 0
        if oh:
            s = -1 if sign == '-' else 1
            tz = timezone(timedelta(hours=int(oh)*s, minutes=int(om)*s))
        else:
            tz = timezone.utc
        return datetime(int(year), int(month), int(day),
                        int(hour), int(minute), int(second), us, tz)

    def _decode_epoch_datetime(self):
        value = self._decode()
        return datetime.fromtimestamp(value, timezone.utc)

    def _decode_positive_bignum(self):
        value = self._decode()
        return int(value.hex(), 16)

    def _decode_negative_bignum(self):
        return -self._decode_positive_bignum() - 1

    def _decode_set(self):
        value = self._decode()
        return frozenset(value) if self._immutable else set(value)

    # ── float デコーダ ────────────────────────────────────────────────────────

    def _decode_float16(self):
        return struct.unpack('>e', self._read(2))[0]

    def _decode_float32(self):
        return struct.unpack('>f', self._read(4))[0]

    def _decode_float64(self):
        return struct.unpack('>d', self._read(8))[0]

    # ── メインエントリポイント ────────────────────────────────────────────────

    def _decode(self):
        """1つのCBOR値を読んで返す"""
        initial_byte = self._read(1)[0]
        major_type = initial_byte >> 5
        subtype    = initial_byte & 0x1F
        return self._major_decoders[major_type](subtype)

    def decode(self):
        """ストリームから1つのCBORオブジェクトをデコードして返す"""
        return self._decode()


def my_loads(data: bytes, **kwargs):
    """CBORバイト列をPythonオブジェクトに変換"""
    return MyCBORDecoder(BytesIO(data), **kwargs).decode()


print("MyCBORDecoder 定義完了")

MyCBORDecoder 定義完了


---
## Step 7: 往復テスト (Round-trip Test)

自作のエンコーダとデコーダが正しく対応しているか確認します。
さらに **cbor2** の公式実装と出力を比較します。

In [30]:
# 往復テスト: encode → decode で元の値に戻るか

import datetime

test_cases = [
    ("整数 0",         0),
    ("整数 23",        23),
    ("整数 24",        24),
    ("整数 1000",      1000),
    ("巨大整数",       2**65),
    ("負の整数",       -100),
    ("True",           True),
    ("False",          False),
    ("None",           None),
    ("float",          3.14),
    ("文字列",         "hello"),
    ("日本語",         "こんにちは"),
    ("バイト列",       b"\x00\x01\x02"),
    ("リスト",         [1, "two", 3.0]),
    ("ネスト",         {"a": [1, 2, {"b": True}]}),
    ("set",            {1, 2, 3}),
]

print(f"{'ラベル':<15} {'元の値':<35} {'encoded (hex)':<30} {'デコード後':<35} {'OK?'}")
print('─' * 120)

all_pass = True
for label, original in test_cases:
    try:
        encoded  = my_dumps(original)
        decoded  = my_loads(encoded)
        # set は list としてデコードされてから set に戻すので比較を調整
        if isinstance(original, set):
            ok = set(decoded) == original
        else:
            ok = decoded == original
        status = '✅' if ok else '❌'
        if not ok: all_pass = False
        print(f"{label:<15} {repr(original):<35} {encoded.hex():<30} {repr(decoded):<35} {status}")
    except Exception as e:
        all_pass = False
        print(f"{label:<15} {repr(original):<35} ERROR: {e}")

print()
print("全テスト合格" if all_pass else "❌ 失敗したテストがあります")

ラベル             元の値                                 encoded (hex)                  デコード後                               OK?
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
整数 0            0                                   00                             0                                   ✅
整数 23           23                                  17                             23                                  ✅
整数 24           24                                  1818                           24                                  ✅
整数 1000         1000                                1903e8                         1000                                ✅
巨大整数            36893488147419103232                c249020000000000000000         36893488147419103232                ✅
負の整数            -100                                3863                           -100                                ✅
True            True          

In [31]:
!pip install cbor2

In [32]:
# cbor2 の公式実装と比較
try:
    import cbor2
    print("cbor2 インポート成功 — 公式実装と比較します")
    print()

    compare_cases = [
        0, 23, 24, 1000, True, False, None,
        3.14, "hello", b"\x01\x02", [1, 2, 3], {"a": 1}
    ]

    print(f"{'値':<25} {'自作 (hex)':<30} {'cbor2 (hex)':<30} {'一致?'}")
    print('─' * 95)

    for obj in compare_cases:
        mine    = my_dumps(obj)
        official = cbor2.dumps(obj)
        match = '✅' if mine == official else '⚠️'
        print(f"{repr(obj):<25} {mine.hex():<30} {official.hex():<30} {match}")

except ImportError:
    print("cbor2 が未インストールです。以下でインストールしてください:")
    print("  pip install cbor2")

cbor2 インポート成功 — 公式実装と比較します

値                         自作 (hex)                       cbor2 (hex)                    一致?
───────────────────────────────────────────────────────────────────────────────────────────────
0                         00                             00                             ✅
23                        17                             17                             ✅
24                        1818                           1818                           ✅
1000                      1903e8                         1903e8                         ✅
True                      f5                             f5                             ✅
False                     f4                             f4                             ✅
None                      f6                             f6                             ✅
3.14                      fb40091eb851eb851f             fb40091eb851eb851f             ✅
'hello'                   6568656c6c6f                   6568656

---
## Step 8: カスタムエンコーダ/デコーダの拡張

cbor2 の `default` / `tag_hook` / `object_hook` パターンを使って、独自の型を拡張する例です。

In [28]:
# ── 例: dataclassのカスタムシリアライズ ──────────────────────────────────────
from dataclasses import dataclass

@dataclass
class Point:
    x: float
    y: float

POINT_TAG = 99999   # 独自タグ番号

# カスタムエンコーダ: Point → CBORTag(99999, [x, y])
def my_default(encoder: MyCBOREncoder, obj):
    if isinstance(obj, Point):
        encoder._encode_semantic(CBORTag(POINT_TAG, [obj.x, obj.y]))
    else:
        raise TypeError(f"エンコード不可: {type(obj).__name__}")

# カスタムデコーダ: CBORTag(99999, [x, y]) → Point
def my_tag_hook(decoder: MyCBORDecoder, tag: CBORTag):
    if tag.tag == POINT_TAG:
        return Point(*tag.value)
    return tag


# エンコード
p = Point(3.0, 4.0)
encoded = my_dumps(p, default=my_default)
print(f"Point エンコード: {encoded.hex()}")

# デコード
decoded = my_loads(encoded, tag_hook=my_tag_hook)
print(f"Point デコード: {decoded}")
print(f"一致: {p == decoded}")

Point エンコード: da0001869f82fb4008000000000000fb4010000000000000
Point デコード: Point(x=3.0, y=4.0)
一致: True


In [29]:
# ── バイト列の詳細可視化ツール ────────────────────────────────────────────────
# CBORバイト列を人間が読みやすい形式で解析表示します。

MAJOR_NAMES = ['uint', 'negint', 'bytes', 'text', 'array', 'map', 'tag', 'special']

def annotate_cbor(data: bytes, indent: int = 0) -> None:
    """CBORバイト列を再帰的にアノテーションして表示"""
    fp = BytesIO(data)
    _annotate(fp, indent)

def _annotate(fp, indent: int, label: str = "") -> None:
    pos = fp.tell()
    b = fp.read(1)
    if not b:
        return
    ib = b[0]
    major = ib >> 5
    sub   = ib & 0x1F
    prefix = '  ' * indent

    if major == 7:  # special
        names = {20:'false', 21:'true', 22:'null', 23:'undefined',
                 25:'float16', 26:'float32', 27:'float64'}
        print(f"{prefix}[0x{ib:02x}] major=7(special) sub={sub} → {names.get(sub, 'simple')}")
        if sub in (25, 26, 27):
            sz = {25:2, 26:4, 27:8}[sub]
            val_bytes = fp.read(sz)
            fmt = {25:'>e', 26:'>f', 27:'>d'}[sub]
            print(f"{prefix}  payload: {val_bytes.hex()} = {struct.unpack(fmt, val_bytes)[0]}")
        return

    # 長さを読む
    if sub < 24: length = sub; extra_bytes = b''
    elif sub == 24: lb=fp.read(1); length=lb[0]; extra_bytes=lb
    elif sub == 25: lb=fp.read(2); length=struct.unpack('>H',lb)[0]; extra_bytes=lb
    elif sub == 26: lb=fp.read(4); length=struct.unpack('>L',lb)[0]; extra_bytes=lb
    elif sub == 27: lb=fp.read(8); length=struct.unpack('>Q',lb)[0]; extra_bytes=lb
    else: length=None; extra_bytes=b''

    mname = MAJOR_NAMES[major]
    print(f"{prefix}[0x{ib:02x}] major={major}({mname}) sub={sub} length/value={length}")

    if major == 0:   pass  # uint: 値が length
    elif major == 1: pass  # negint
    elif major in (2, 3):  # bytes or text
        payload = fp.read(length)
        if major == 3:
            print(f"{prefix}  → {payload.decode('utf-8')!r}")
        else:
            print(f"{prefix}  → {payload.hex()}")
    elif major == 4:  # array
        print(f"{prefix}  [{length} items]")
        rest = fp.read()
        # 単純のため残りを渡す（本来はfpを共有すべき）
    elif major == 6:  # tag
        print(f"{prefix}  tagged value:")


# 例示
print("=== annotate: 整数 1000 ===")
annotate_cbor(my_dumps(1000))

print("\n=== annotate: 文字列 'hello' ===")
annotate_cbor(my_dumps("hello"))

print("\n=== annotate: float 3.14 ===")
annotate_cbor(my_dumps(3.14))

=== annotate: 整数 1000 ===
[0x19] major=0(uint) sub=25 length/value=1000

=== annotate: 文字列 'hello' ===
[0x65] major=3(text) sub=5 length/value=5
  → 'hello'

=== annotate: float 3.14 ===
[0xfb] major=7(special) sub=27 → float64
  payload: 40091eb851eb851f = 3.14


---
## まとめ

このノートブックで実装した内容:

| Step | 内容 |
|---------|------|
| 1 | `CBORTag`, `CBORSimpleValue`, `FrozenDict`, `undefined` などの型部品 |
| 2 | `encode_length` — CBORの初期バイト生成の核心ロジック |
| 3 | 整数・文字列・バイト列・浮動小数点の個別エンコーダ |
| 4 | `MyCBOREncoder` — 型ディスパッチを持つクラスに組み上げ |
| 5 | `_decode_length` — initial byte の解析ロジック |
| 6 | `MyCBORDecoder` — major type / semantic tag テーブル駆動設計 |
| 7 | 往復テスト (round-trip) と cbor2 公式実装との比較 |
| 8 | カスタム型の拡張パターン (`default` / `tag_hook`) |